In [ ]:

# 4A: CBOW for Document Classification using Deep Learning

# 1. Import libraries
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam


# 2. Load dataset
data = pd.read_csv("IMDB Dataset.csv")

print("Dataset Shape:", data.shape)
print(data.head())

# Remove missing values
data = data.dropna(subset=['review', 'sentiment'])

# Convert sentiment into numbers
data['sentiment'] = data['sentiment'].str.lower().map({
    'positive': 1,
    'negative': 0
})

data = data.dropna(subset=['sentiment'])

X = data['review'].astype(str)
y = data['sentiment'].astype(int)


# 3. Text preprocessing
def preprocess(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    return tokens

tokenized_reviews = X.apply(preprocess).tolist()

print("\nSample Tokens:")
print(tokenized_reviews[0][:15])


# 4. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    tokenized_reviews,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))


# 5. Train CBOW Word2Vec model
cbow_model = Word2Vec(
    sentences=X_train,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,
    workers=4,
    epochs=10
)

print("\nCBOW Model Training Completed!")
print("Vocabulary Size:", len(cbow_model.wv))


# 6. Convert each review into a document vector
def get_document_vector(tokens):

    vectors = [
        cbow_model.wv[word]
        for word in tokens
        if word in cbow_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


# Convert training and testing reviews
X_train_vectors = np.array([
    get_document_vector(tokens) for tokens in X_train
])

X_test_vectors = np.array([
    get_document_vector(tokens) for tokens in X_test
])

y_train = np.array(y_train)
y_test = np.array(y_test)

print("\nTraining Vector Shape:", X_train_vectors.shape)
print("Testing Vector Shape:", X_test_vectors.shape)


# 7. Build Deep Learning model
model = Sequential([
    Dense(64, activation='relu', input_shape=(100,)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# 8. Compile model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


# 9. Train model
history = model.fit(
    X_train_vectors,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)


# 10. Evaluate model
loss, accuracy = model.evaluate(X_test_vectors, y_test)

print("\nTest Loss:", loss)
print("Test Accuracy:", accuracy)


# 11. Predict sentiment
predictions = model.predict(X_test_vectors)

y_pred = (predictions > 0.5).astype(int).flatten()

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Negative', 'Positive']
))


# 12. Plot training and validation accuracy
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


# 13. Plot training and validation loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


# 14. Test with a new review
review = input("\nEnter a movie review: ")

tokens = preprocess(review)

review_vector = get_document_vector(tokens)

prediction = model.predict(
    np.array([review_vector]),
    verbose=0
)[0][0]

print("\nPredicted Sentiment:")

if prediction >= 0.5:
    print("Positive")
else:
    print("Negative")
